In [ ]:
from dbrepo.RestClient import RestClient
import pandas as pd
from dotenv import load_dotenv
import os 

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [ ]:
DB_ID = os.getenv("DB_ID")
df = client.get_database(DB_ID)

## Check all views

In [8]:
def get_joined_view_id(df):
    """Automate the view id lookup"""
    for t in df.views:
        if t.name == "drug_gdp_features_view":
            return t.id 
        
for t in df.views:
    print(t.name, t.id)

drug_gdp_features_view 6a6080f4-4117-4201-af05-876bf9eb05d5
ww_city_year_drug_summary 8550c148-db32-475c-8be6-d55e34782949


## Import view from API

In [9]:
db_id = DB_ID
view_id = get_joined_view_id(df)

response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}"
)

print(response.status_code)
db = response.json()

200


In [10]:
print(db)

{'id': '6a6080f4-4117-4201-af05-876bf9eb05d5', 'name': 'drug_gdp_features_view', 'identifiers': [], 'query': 'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`', 'owner': {'id': None, 'username': 'data_st

In [24]:
dict(db)["query"]

'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`'

In [19]:
q = dict(db)["query"]

In [23]:
q.replace("dast_g20_wastewater_epidemiology_ndfx", "db").replace("wastewater_data", "ww").split("from")

['select `db`.`ww`.`daily_mean_concentration` as `daily_mean`, `db`.`ww`.`metabolite_name` as `metabolite_name`, `db`.`ww`.`ref_year` as `ref_year`, `db`.`city_map`.`nuts_code` as `nuts_code`, `db`.`ww`.`city_name` as `city_name`, `db`.`gdp_data`.`gdp` as `gdp` ',
 ' `ww` join `city_map` on `db`.`ww`.`city_name` = `db`.`city_map`.`city_name` join `gdp_data` on `db`.`gdp_data`.`nuts_code` = `db`.`city_map`.`nuts_code`']

In [11]:
response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data",
    headers={"Accept": "application/json"}
)
print(response.status_code)
print(response.json())

200
[{'city_name': 'Purgstall', 'daily_mean': 22.03, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 4.64, 'gdp': 6574580000.0, 'metabolite_name': 'MDMA', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 13.74, 'gdp': 6574580000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 1.48, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 30.6, 'gdp': 6574580000.0, 'metabolite_name': 'cannabis', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 24.39, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 1.32, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2020}, {

In [12]:
print(len(response.json()))

10


In [ ]:
# get_view_data(self, database_id: str, view_id: str, page: int = 0, size: int = 1000000) -> DataFrame:
#         """
#         Get data of a view in a database with the given database id and view id.

#         :param database_id: The database id.
#         :param view_id: The view id.
#         :param page: The result pagination number. Optional. Default: `0`.
#         :param size: The result pagination size. Optional. Default: `1000000`.

#         :returns: The view data, if successful.

#         """
#         url = f'/api/v1/database/{database_id}/view/{view_id}/data'
#         params = []
#         if page is not None and size is not None:
#             params.append(('page', page))
#             params.append(('size', size))
#         response = self._wrapper(method="get", url=url, params=params, headers={'Accept': 'application/json'})


response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data?limit=50",
    headers={"Accept": "application/json"}
)

print(response.status_code)
print(response.json())
print(len(response.json()))

200
[{'city_name': 'Purgstall', 'daily_mean': 22.03, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 4.64, 'gdp': 6574580000.0, 'metabolite_name': 'MDMA', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 13.74, 'gdp': 6574580000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 1.48, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 30.6, 'gdp': 6574580000.0, 'metabolite_name': 'cannabis', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 24.39, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 1.32, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2020}, {

In [14]:
fetched_data = client.get_view_data(database_id = DB_ID,
                                        view_id=view_id)
#,page: int = 0, size: int = 1000000)

In [ ]:
fetched_data.drop_duplicates() # join did not have the ref_year condition -> incorrect view

,city_name,daily_mean,gdp,metabolite_name,nuts_code,ref_year
0,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2019
1,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2016
2,Amsterdam,1142.43,1.072809e+10,cocaine,NL321,2022
3,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2017
4,Innsbruck,17.05,1.373652e+10,amphetamine,AT332,2018
...,...,...,...,...,...,...
37971,Kufstein,20.67,1.399246e+10,amphetamine,AT335,2023
37972,Santiago,64.90,2.014030e+10,cannabis,ES114,2018
37973,Santiago,30.41,2.172080e+10,MDMA,ES114,2022
37974,Rovaniemi,1.87,6.771740e+09,methamphetamine,FI1D7,2020


## Transform into dataset, ready to be used

## Test if results stay the same